# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!pip install -q gdown

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
import pandas as pd
from pathlib import Path
import gdown

# Create data directory if it doesn't exist
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

# Dataset path
dataset_path = data_dir / "content_refresh_anonymized.csv"

# Download dataset only if it doesn't exist
if not dataset_path.exists():
    file_id = "1d9-pVHOqtW3ONxLgKFYxjp0zPsrumEh7"
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        str(dataset_path),
        quiet=False
    )

# Load dataset
df = pd.read_csv(dataset_path)

print(f"Dataset loaded successfully: {df.shape}")
display(df.head())

# ---------- Signal 1 : Staleness ----------
bins = [0, 30, 90, 180, 365, 10000]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]

df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=bins,
    labels=labels
)

signal1 = (
    df.groupby("stale_bucket", observed=False)
      .agg(
          avg_health=("health_score", "mean"),
          refresh_rate=("is_initial_refresh_candidate", "mean"),
          n=("content_id", "count")
      )
)

print("Signal 1: Staleness")
display(signal1)

print(
    "Verdict:",
    "CONFIRMED"
    if signal1["refresh_rate"].iloc[-1] > signal1["refresh_rate"].iloc[0]
    else "MIXED"
)

# ---------- Signal 2 : CTR vs Position ----------
bins = [0, 3, 5, 10, 20, 100]
labels = ["1-3", "4-5", "6-10", "11-20", "20+"]

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=bins,
    labels=labels
)

signal2 = (
    df.groupby("position_bucket", observed=False)
      .agg(
          avg_ctr=("ctr", "mean"),
          ctr_fix_rate=("needs_ctr_fix", "mean"),
          n=("content_id", "count")
      )
)

print("\nSignal 2: CTR vs Position")
display(signal2)

print(
    "Verdict:",
    "CONFIRMED"
    if signal2["avg_ctr"].iloc[0] > signal2["avg_ctr"].iloc[-1]
    else "MIXED"
)

Dataset loaded successfully: (30000, 53)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


Signal 1: Staleness


,avg_health,refresh_rate,n
stale_bucket,,,
0-30,41.948730,0.318373,20480
31-90,37.971429,0.508571,175
91-180,42.087013,0.518918,9171
181-365,43.905325,0.12426,169
365+,38.000000,0.0,5


Verdict: MIXED

Signal 2: CTR vs Position


,avg_ctr,ctr_fix_rate,n
position_bucket,,,
1-3,2.714303,0.069238,1141
4-5,1.104820,0.035226,2782
6-10,0.511708,0.039735,9060
11-20,0.323443,0.000000,7273
20+,0.211705,0.000000,8524


Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
# Baseline score
df["baseline_action_score"] = (
      (df["days_since_last_update"] > 180).astype(int) * 40
    + (df["ctr"] < 0.03).astype(int) * 30
    + (df["search_volume"] > 1000).astype(int) * 20
    + (df["avg_position"] <= 10).astype(int) * 10
)

# Reason code
def reason(row):
    if row["days_since_last_update"] > 180:
        return "STALE_REFRESH"
    elif row["ctr"] < 0.03:
        return "LOW_CTR"
    elif row["search_volume"] > 1000:
        return "HIGH_VOLUME"
    return "MONITOR"

df["reason_code"] = df.apply(reason, axis=1)

# Action label
action_map = {
    "STALE_REFRESH": "Refresh Content",
    "LOW_CTR": "Improve CTR",
    "HIGH_VOLUME": "Quick Win",
    "MONITOR": "Monitor"
}

df["action_label"] = df["reason_code"].map(action_map)

# Rank
ranked = df.sort_values(
    "baseline_action_score",
    ascending=False
)

# Export CSV
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

ranked.to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

display(
    ranked[
        [
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)


CSV written successfully.


,content_id,baseline_action_score,reason_code,action_label
1659,content_bbca724138f2,90,STALE_REFRESH,Refresh Content
7719,content_f783292bc4a0,80,STALE_REFRESH,Refresh Content
24216,content_1b4ec72dafd4,80,STALE_REFRESH,Refresh Content
24557,content_84d12054c0c0,80,STALE_REFRESH,Refresh Content
29530,content_573248f65db2,80,STALE_REFRESH,Refresh Content
24603,content_06f581dd9383,80,STALE_REFRESH,Refresh Content
3429,content_7b4e68b406b8,80,STALE_REFRESH,Refresh Content
14545,content_e2fb3f55bed3,80,STALE_REFRESH,Refresh Content
20924,content_277eeb6d46cc,80,STALE_REFRESH,Refresh Content
3651,content_fd16e3475c29,80,STALE_REFRESH,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked.head(20)

for i, row in enumerate(top20.itertuples(), start=1):
    print(f"{i}. Action: {row.action_label}")
    print(f"   Reason Code: {row.reason_code}")
    print(f"   Confidence: High score based on baseline rule.")
    print("   What would make it wrong? Seasonal demand, missing analytics, or recent unpublished improvements.")
    print("-" * 70)


1. Action: Refresh Content
   Reason Code: STALE_REFRESH
   Confidence: High score based on baseline rule.
   What would make it wrong? Seasonal demand, missing analytics, or recent unpublished improvements.
----------------------------------------------------------------------
2. Action: Refresh Content
   Reason Code: STALE_REFRESH
   Confidence: High score based on baseline rule.
   What would make it wrong? Seasonal demand, missing analytics, or recent unpublished improvements.
----------------------------------------------------------------------
3. Action: Refresh Content
   Reason Code: STALE_REFRESH
   Confidence: High score based on baseline rule.
   What would make it wrong? Seasonal demand, missing analytics, or recent unpublished improvements.
----------------------------------------------------------------------
4. Action: Refresh Content
   Reason Code: STALE_REFRESH
   Confidence: High score based on baseline rule.
   What would make it wrong? Seasonal demand, missing an

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Lowest-scoring pages")
display(
    ranked[
        [
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action_label"
        ]
    ].tail(10)
)

print("\nLeakage Check")

leak_columns = [
    "is_initial_refresh_candidate",
    "needs_ctr_fix",
    "is_quick_win",
    "is_declining"
]

used = [
    "days_since_last_update",
    "ctr",
    "search_volume",
    "avg_position"
]

print("Columns used for scoring:", used)
print("Product flags available:", leak_columns)
print("No product flags or future-window labels were used in the baseline score.")


Lowest-scoring pages


,content_id,baseline_action_score,reason_code,action_label
2213,content_26ce4f120d2e,0,MONITOR,Monitor
2221,content_5b67710ced1c,0,MONITOR,Monitor
29975,content_bdee2164f576,0,MONITOR,Monitor
29976,content_851c604b0631,0,MONITOR,Monitor
4328,content_51b2c4609daf,0,MONITOR,Monitor
4334,content_6b9f8824e0d7,0,MONITOR,Monitor
29958,content_d6ce4e17f464,0,MONITOR,Monitor
29964,content_07ea0872b973,0,MONITOR,Monitor
29966,content_77867ed726e1,0,MONITOR,Monitor
18,content_0b360eb9db55,0,MONITOR,Monitor



Leakage Check
Columns used for scoring: ['days_since_last_update', 'ctr', 'search_volume', 'avg_position']
Product flags available: ['is_initial_refresh_candidate', 'needs_ctr_fix', 'is_quick_win', 'is_declining']
No product flags or future-window labels were used in the baseline score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.